In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install spatialdata "anndata==0.12.2" "scanpy==1.11.4" "squidpy==1.6.5"

# Train on mouse brain data



## import

In [3]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

GPU:  Tesla T4


In [4]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Steamboat_X/examples_m_integrated_1_d")
cwd = os.getcwd()
print(cwd)

sys.path.append("../")

/content/drive/MyDrive/Thesis/Projects/Steamboat_X/examples_m_integrated_1_d


In [5]:
import scanpy as sc
import squidpy as sq
from anndata import AnnData
import spatialdata as sd
import pandas as pd
from tqdm.notebook import tqdm
import scipy as sp
import numpy as np
import multiprocessing
import pickle as pkl
import torch
import gc
import sklearn.metrics
from sklearn.preprocessing import Normalizer, StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)
/usr/local/lib/python3.12/dist-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [6]:
import steamboat_m_integrated_1_d as sf
#importlib.reload(sm)
#import steamboat.integrated_model
# importlib.reload(spaceformer.benchmarks)

In [7]:
import importlib
import steamboat_m_integrated_1_d.tools
import steamboat_m_integrated_1_d.model

## Creat Anndata

In [ ]:
Xenium_path = "/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr"
sdata = sd.read_zarr(Xenium_path)
sdata

In [ ]:
adata = sdata.tables["table"]
adata

In [ ]:
####SINA

adata.shape      # (cells, genes)
#adata.n_vars

In [ ]:
adata_omiCLIP = sc.read_h5ad("../../Data/Mouse_Coronal_Embeddings/cells.h5ad")
adata_omiCLIP

In [ ]:
# 1. Ensure cell IDs are the index (not just a column)
if 'cell_id' in adata.obs.columns:
    adata.obs.set_index('cell_id', inplace=True)
if 'cell_id' in adata_omiCLIP.obs.columns:
    adata_omiCLIP.obs.set_index('cell_id', inplace=True)

# 2. Align the two objects by cell_id (intersection)
common_ids = adata.obs_names.intersection(adata_omiCLIP.obs_names)

# Optional: check how many matched
print(f"Matched {len(common_ids)} cells out of {adata.n_obs}")

# 3. Reorder both to the same order
adata_c = adata[common_ids, :].copy()
adata_omiCLIP_c = adata_omiCLIP[common_ids, :].copy()

# 4. Add the X_custom matrix to adata_main.obsm
adata_c.obsm['Morpho_Embedding'] = adata_omiCLIP_c.obsm['X_custom']

# 5. Done! Verify
print(adata_c.obsm.keys())

In [ ]:
adata_c.write("../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

## Train

In [8]:
adata = sc.read_h5ad("../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

Founsation Models

In [9]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/h_optimus.h5ad")
M = edata.obsm['h_optimus']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/UNI.h5ad")
M = edata.obsm['UNI']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [9]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/virchow.h5ad")
M = edata.obsm['virchow']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [11]:
edata.obs_names = [cid[61:] for cid in edata.obs['cell_id']]
common_cells = ad.obs_names.intersection(edata.obs_names)
ad.obsm['morpho'][ad.obs_names.get_indexer(common_cells)] = \
  edata.obsm['virchow'][edata.obs_names.get_indexer(common_cells)]

In [12]:
#normalizer = Normalizer(norm="l2")
#ad.obsm['morpho'] = normalizer.transform(ad.obsm['morpho'])

adata.obsm['p_Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

Preparing Dataset

In [13]:
adatas = []
for i in adata.obs['region'].unique():
    adatas.append(adata[adata.obs['region'] == i])
    adatas[-1].obs['global'] = 0  #Only support one unique value for regional observation.

/tmp/ipython-input-4759537.py:4: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adatas[-1].obs['global'] = 0  #Only support one unique value for regional observation.


In [14]:
adatas = sf.prep_adatas(adatas, norm=True, log1p=True)

  0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
dataset = sf.make_dataset(adatas, sparse_graph=True, regional_obs=['global'])

Using None to mask variables. Explicitly specify `mask_var=False` to use all genes.
Using ['global'] as regional annotations.


  0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
print('Expression', dataset[0][0].shape)
print('Morpho_Embedding', dataset[0][1].shape)
print('Neighborhood_Graph', dataset[0][2].shape)
print('Regional_Expression', dataset[0][3][0].shape)
print('Regional_Morpho', dataset[0][4][0].shape)
print('Regional_Neighbors', dataset[0][5][0].shape)

Expression torch.Size([63173, 2000])
Morpho_Embedding torch.Size([63173, 2560])
Neighborhood_Graph torch.Size([2, 505384])
Regional_Expression torch.Size([1, 2000])
Regional_Morpho torch.Size([1, 2560])
Regional_Neighbors torch.Size([2, 63173])


In [20]:
importlib.reload(sf.dataset)
importlib.reload(sf.model)
importlib.reload(sf)

<module 'steamboat_m_integrated_1_d' from '/content/drive/MyDrive/Thesis/Projects/Steamboat_X/examples_m_integrated_1_d/../steamboat_m_integrated_1_d/__init__.py'>

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [17]:
sf.set_random_seed(0)
model = sf.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, n_scales=3)
model = model.to(device)
# model.load_state_dict(torch.load('saved_models/mmbrain_new.pth', weights_only=True))


In [18]:
model.fit(dataset, entry_masking_rate=0.3, feature_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched = None,
          max_lr=0.01, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=200, stop_tol=200)

DP koon


[2026-01-22 13:55:41,164::train::INFO] Epoch 1: train_loss 1662.61426
INFO:train:Epoch 1: train_loss 1662.61426
[2026-01-22 13:59:42,135::train::INFO] Epoch 201: train_loss 1551.02039
INFO:train:Epoch 201: train_loss 1551.02039
[2026-01-22 14:03:38,879::train::INFO] Epoch 401: train_loss 1335.55542
INFO:train:Epoch 401: train_loss 1335.55542
[2026-01-22 14:07:34,429::train::INFO] Epoch 601: train_loss 1178.62842
INFO:train:Epoch 601: train_loss 1178.62842
[2026-01-22 14:11:30,595::train::INFO] Epoch 801: train_loss 1043.76013
INFO:train:Epoch 801: train_loss 1043.76013
[2026-01-22 14:15:27,427::train::INFO] Epoch 1001: train_loss 923.70300
INFO:train:Epoch 1001: train_loss 923.70300
[2026-01-22 14:19:25,564::train::INFO] Epoch 1201: train_loss 815.47705
INFO:train:Epoch 1201: train_loss 815.47705
[2026-01-22 14:23:23,064::train::INFO] Epoch 1401: train_loss 717.46631
INFO:train:Epoch 1401: train_loss 717.46631
[2026-01-22 14:27:20,228::train::INFO] Epoch 1601: train_loss 628.61554
INFO

Steamboat(
  (spatial_gather): BilinearAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2560, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=256, bias=True)
        (3): ReLU()
        (4): Linear(in_features=256, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_Morpho): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_regionals): ModuleList(
      (0): NonNegLinear(
        (elu): ELU(alpha=1.0)
      )
    )
    (w_ego): NonNegScale(
      (elu): ELU(alpha=1.0)
    )
    (w_Morpho): NonNegScale(
      (elu): ELU(alpha=1.0)
    )
    (w_local): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (w_global): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (tanh): Tanh()
    (v): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (v_m)

In [ ]:
"""
Input and Output of the model:

Input:

Graph structure (adj_list, regional_adj_lists)

Cell features (x, masked_x)

Regional features (regional_xs)

Output:

Reconstructed cell features ([n_cells, n_genes])

attention weights + embeddings (dict)

"""


In [19]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), 'saved_models/Xmbrain_50_0.3_virchow.pth')
else:
    model.load_state_dict(torch.load('saved_models/Xmbrain_50_0.3_hoptimus.pth'))